In [1]:
from transformers import DistilBertTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# Load pre-trained BERT model fine-tuned for sentiment classification
model_name = 'tabularisai/multilingual-sentiment-analysis' # Pretrained model finetuned on multilingual languages for sentiment analysis; based on DistillBERT
                                                           # You can use other pretrained models if you like  

tokenizer = DistilBertTokenizer.from_pretrained(model_name) # In Hugging Face, models and tokenizers are kept as separate objects. 
model = AutoModelForSequenceClassification.from_pretrained(model_name) # Bert Model transformer with a sequence classification head on top (a linear layer on top of the pooled output)
model.eval() # Evaluation mode

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)

In [2]:
# For this model: 0 = very negative, ..., 4 = very positive
label_map = {
    0: "very negative",
    1: "negative",
    2: "neutral",
    3: "positive",
    4: "very positive"
}

In [3]:
# Method for generating sentiment predictions; fill in missing lines of codes
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # Disable gradient calculations for inference
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True) #output_hidden_states to reveal contextual embeddings

    # Get the logits
    logits = outputs.logits

    # Apply softmax to get probabilities
    probabilities = torch.softmax(logits, dim=1)

    # 5. Insert lines to get the hidden states
    hidden_states = outputs.hidden_states

    # Get the predicted class (index with the highest probability)
    predicted_class_id = torch.argmax(probabilities, dim=1).item()
    print(f'predicted_class_id: {predicted_class_id}')

    # Get the confidence score
    confidence = probabilities[0][predicted_class_id].item()

    # Map the predicted class ID to a sentiment label
    predicted_sentiment = label_map.get(predicted_class_id, "Unknown")

    print(f"Text: \"{text}\"")
    print(f"Predicted Sentiment: {predicted_sentiment} (Confidence: {confidence:.4f})")
    
    # 6. Show the resulting logits
    print(f"Logits: {logits}")
    
    # 7. Show the resulting probabilities
    print(f"Probabilities: {probabilities}")
    
    # 8. Show the shape of the hidden states
    print(f"Number of hidden-state layers: {len(hidden_states)}")
    print(f"Hidden states shape: {hidden_states[-1].shape}")

    print("-" * 30)
    print("-" * 30)

In [4]:
# Test cases
predict_sentiment("This movie was an absolute masterpiece! I loved every single moment of it.")
# The sentiment here was correct and true. THis is a positive sentiment which has a class of 4.
predict_sentiment("I can't believe how bad this film was. A complete waste of time and money.")
# The sentiment here was also correct and true. This is a negative sentiment which has a class of 0.
predict_sentiment("The movie was alright, nothing special.")
# The sentiment here was only partly true. THis can be a positive sentiment, however, in my opinion
# this sentiment was a neutral one so it should have had a class of 2.
predict_sentiment("Ang ganda. Nakakasuka!")
# The sentiment here was also inaccurate. This is expected since the sentiment was in Tagalog, a language
# Bert was not really fine tuned on. "Nakakasuka" is actually a negative sentiment, this was probably sarcasm.

# Hence, this model predicts very accurately on both edges (0 and 4) especially on English, but fails on midpoint 
#or on languages it has not been finetuned on.

predicted_class_id: 4
Text: "This movie was an absolute masterpiece! I loved every single moment of it."
Predicted Sentiment: very positive (Confidence: 0.8340)
Logits: tensor([[-1.4717, -1.4603, -1.0171,  0.4027,  2.4552]])
Probabilities: tensor([[0.0164, 0.0166, 0.0259, 0.1071, 0.8340]])
Number of hidden-state layers: 7
Hidden states shape: torch.Size([1, 18, 768])
------------------------------
------------------------------
predicted_class_id: 0
Text: "I can't believe how bad this film was. A complete waste of time and money."
Predicted Sentiment: very negative (Confidence: 0.9138)
Logits: tensor([[ 2.8546, -0.2607, -1.2015, -1.2523, -1.2710]])
Probabilities: tensor([[0.9138, 0.0405, 0.0158, 0.0150, 0.0148]])
Number of hidden-state layers: 7
Hidden states shape: torch.Size([1, 21, 768])
------------------------------
------------------------------
predicted_class_id: 3
Text: "The movie was alright, nothing special."
Predicted Sentiment: positive (Confidence: 0.6236)
Logits: tensor(

## AI Use Declaration
- I hereby declare upon my honor that I only used AI to understand the codes and concepts in this exercise.
- I did not use AI to generate any answers. All answers were written by me and are my own work.